# Serie I: modelo para planificar la siguiente zafra

**Análisis revisado: particiones temporales, historia anterior al año predicho y métricas reconstruidas con pruebas.** Se predice azúcar recuperable por acre (`T_SPACRE`, lb/acre) para un año posterior, con variedad, localidad, etapa, corte planificado e información histórica disponible hasta el año anterior. Se comparan tres algoritmos y una referencia histórica sencilla.

La validación simula dos pronósticos de un año: entrenamiento hasta 2018 y validación en 2019; entrenamiento hasta 2019 y validación en 2020. Se elige por MAE promedio de esas validaciones. El año 2021 queda reservado para la evaluación final, sin usarlo para elegir algoritmo. Las filas de una misma combinación en un año nunca se reparten entre entrenamiento y evaluación.

`TCA`, `TRS_TON`, `POPN`, `MSTWT` y `FIBER` de la zafra que se quiere predecir no se usan como entradas. En particular, `TCA × TRS_TON ≈ T_SPACRE`: usarlas daría una precisión poco representativa de la planificación. Los promedios históricos del objetivo se calculan con años estrictamente anteriores; el procesamiento y la imputación se ajustan solo con el entrenamiento de cada corte temporal.

El historial, las referencias y los pipelines están en `scripts/modelado_serie1.py`, compartidos con pytest. Se guardan predicciones por fila para 2019, 2020 y los cinco métodos evaluados en 2021. Las pruebas de integración reconstruyen sus métricas; las unitarias comprueban que modificar objetivos presentes/futuros no altere la historia disponible y que la imputación/codificación/escalado use solo entrenamiento. Estas comprobaciones verifican implementación y trazabilidad, no garantizan precisión en otra zafra.


In [1]:
import sys
print("Python de ejecución:", sys.executable)
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.tree import export_text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUTA = Path("../data/processed/serie1_etl_provisional.csv")
if not RUTA.exists():
    RUTA = Path("data/processed/serie1_etl_provisional.csv")
RAIZ = RUTA.resolve().parents[2]
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from scripts.modelado_serie1 import CATEGORICAS, ENTRADAS, agregar_historia, referencia_reciente, crear_modelos

# El notebook y pytest utilizan las mismas funciones y pipelines.
datos = pd.read_csv(RUTA, low_memory=False)
excluir = datos["REQUIERE_REVISION"] | datos["T_SPACRE"].isna()
excluidas = datos.loc[excluir].copy()
excluidas["MOTIVO_EXCLUSION_ANALISIS"] = np.where(
    excluidas["T_SPACRE"].isna(), "Falta variable objetivo.", "Alerta de calidad; sensibilidad con inclusión disponible."
)
excluidas.to_csv(RUTA.parent / "serie1_modelado_exclusiones.csv", index=False)
base = datos.loc[~excluir].copy()
print(f"Filas fuente analítica: {len(datos):,}; modelado: {len(base):,}; excluidas temporalmente: {len(excluidas):,}.")
print(f"Ceros fuente en el objetivo conservados: {base['T_SPACRE'].eq(0).sum()}.")
modelado = agregar_historia(base)
categoricas = CATEGORICAS
entradas = ENTRADAS
modelado[categoricas] = modelado[categoricas].astype(str)
algoritmos = crear_modelos()
print("Variables disponibles antes de la zafra:", entradas)


Python de ejecución: C:\Users\user\Desktop\Cursos\cesar\otros\yo\Prueba Santa Ana\.venv\Scripts\python.exe


Filas fuente analítica: 29,403; modelado: 28,981; excluidas temporalmente: 422.
Ceros fuente en el objetivo conservados: 8.


Variables disponibles antes de la zafra: ['CLONE', 'LOC', 'STAGE', 'YEAR', 'CROP', 'HIST_CLONE', 'HIST_LOC', 'HIST_CLONE_CROP']


## Comparación temporal y elección

Se mantienen hiperparámetros sencillos y fijos; no se hace una búsqueda extensa. Ridge representa una relación aditiva regularizada; el árbol produce reglas; el bosque permite interacciones y relaciones no lineales. La referencia predice con la media histórica de clon/corte; si no existe, usa clon, localidad y finalmente la media de entrenamiento.

MAE y RMSE se expresan en lb de azúcar/acre. Se reportan ambas: MAE para el error típico de planificación y RMSE para penalizar errores grandes. R² complementa la lectura sin sustituir la comparación contra la referencia.

Se añade una segunda referencia más exigente: el mismo clon, localidad, etapa y corte del año anterior. Si no hay esa observación, usa la historia disponible de esa combinación, luego etapa/corte y finalmente el promedio de entrenamiento.

In [2]:
resultados = []
predicciones_validacion = []
for anio in [2019, 2020]:
    entrenamiento = modelado.loc[modelado["YEAR"].lt(anio)]
    evaluacion = modelado.loc[modelado["YEAR"].eq(anio)]
    referencia = evaluacion["HIST_CLONE_CROP"].fillna(evaluacion["HIST_CLONE"]).fillna(evaluacion["HIST_LOC"]).fillna(entrenamiento["T_SPACRE"].mean())
    predicciones = {"Referencia histórica": referencia.to_numpy(), "Referencia clon-ambiente reciente": referencia_reciente(entrenamiento, evaluacion).to_numpy()}
    for nombre, algoritmo in algoritmos.items():
        modelo = clone(algoritmo)
        modelo.fit(entrenamiento[entradas], entrenamiento["T_SPACRE"])
        predicciones[nombre] = modelo.predict(evaluacion[entradas])
    for nombre, prediccion in predicciones.items():
        predicciones_validacion.append(pd.DataFrame({
            "FILA_CSV": evaluacion["FILA_CSV"].to_numpy(), "OBSERVADO": evaluacion["T_SPACRE"].to_numpy(),
            "PREDICCION": prediccion, "MODELO": nombre, "ANIO_EVALUACION": anio,
            "MAX_ANIO_ENTRENAMIENTO": int(entrenamiento["YEAR"].max()),
        }))
        resultados.append({
            "ANIO_VALIDACION": anio, "MODELO": nombre,
            "MAX_ANIO_ENTRENAMIENTO": int(entrenamiento["YEAR"].max()),
            "MAE": mean_absolute_error(evaluacion["T_SPACRE"], prediccion),
            "RMSE": np.sqrt(mean_squared_error(evaluacion["T_SPACRE"], prediccion)),
            "SESGO": np.mean(prediccion - evaluacion["T_SPACRE"].to_numpy()),
            "R2": r2_score(evaluacion["T_SPACRE"], prediccion),
            "N": len(evaluacion),
        })
validacion = pd.DataFrame(resultados)
display(validacion.round(2))
niveles_zafra = modelado.loc[modelado["YEAR"].isin([2018, 2019, 2020])].pivot_table(
    index="STAGE", columns="YEAR", values="T_SPACRE", aggfunc="mean"
)
niveles_zafra["CAMBIO_2018_2019_%"] = 100 * (niveles_zafra[2019] / niveles_zafra[2018] - 1)
niveles_zafra["CAMBIO_2019_2020_%"] = 100 * (niveles_zafra[2020] / niveles_zafra[2019] - 1)
print("Promedio del objetivo por etapa y variación interanual en filas modeladas:")
display(niveles_zafra.round(1))
comparacion = validacion.groupby("MODELO")[["MAE", "RMSE", "R2"]].mean().sort_values("MAE")
display(comparacion.round(2))
elegido = comparacion.loc[comparacion.index.isin(algoritmos)].index[0]
print("Algoritmo elegido por validación 2019–2020:", elegido)
validacion.to_csv(RUTA.parent / "serie1_validacion_modelos.csv", index=False)
pd.concat(predicciones_validacion, ignore_index=True).to_csv(RUTA.parent / "serie1_predicciones_validacion.csv", index=False)


,ANIO_VALIDACION,MODELO,MAX_ANIO_ENTRENAMIENTO,MAE,RMSE,SESGO,R2,N
0,2019,Referencia histórica,2018,3088.23,3815.64,2804.57,-1.59,2304
1,2019,Referencia clon-ambiente reciente,2018,2699.06,3341.71,2052.18,-0.99,2304
2,2019,Ridge interpretable,2018,2515.28,3116.38,2147.76,-0.73,2304
3,2019,Árbol de decisión,2018,2602.75,3188.76,2119.90,-0.81,2304
4,2019,Bosque aleatorio,2018,2743.86,3344.82,2188.10,-0.99,2304
5,2020,Referencia histórica,2019,2386.03,3209.40,-267.26,0.02,2207
6,2020,Referencia clon-ambiente reciente,2019,2752.09,3502.02,-1509.09,-0.16,2207
7,2020,Ridge interpretable,2019,2296.01,3076.49,-1078.48,0.10,2207
8,2020,Árbol de decisión,2019,2973.15,3789.16,-2052.84,-0.36,2207
9,2020,Bosque aleatorio,2019,3021.11,3795.28,-2163.30,-0.36,2207


Promedio del objetivo por etapa y variación interanual en filas modeladas:


YEAR,2018,2019,2020,CAMBIO_2018_2019_%,CAMBIO_2019_2020_%
STAGE,,,,,
3,11138.8,8246.9,11651.1,-26.0,41.3
4,9464.1,7517.8,10473.1,-20.6,39.3
5,9200.7,6726.6,8770.9,-26.9,30.4


,MAE,RMSE,R2
MODELO,,,
Ridge interpretable,2405.64,3096.43,-0.31
Referencia clon-ambiente reciente,2725.58,3421.87,-0.58
Referencia histórica,2737.13,3512.52,-0.78
Árbol de decisión,2787.95,3488.96,-0.59
Bosque aleatorio,2882.48,3570.05,-0.68


Algoritmo elegido por validación 2019–2020: Ridge interpretable


## Evaluación reservada: zafra 2021

Se entrena hasta 2020 y se evalúa en 2021. Se conserva la elección previa aunque otro algoritmo resulte mejor en este año; no se utiliza 2021 para retocar hiperparámetros. Además del resultado global se informa el error en etapa 5 y en clones vistos/no vistos. La validación representa el siguiente año de los ambientes del archivo, no una finca nueva sin historia.

In [3]:
entrenamiento = modelado.loc[modelado["YEAR"].le(2020)]
prueba = modelado.loc[modelado["YEAR"].eq(2021)].copy()
modelos_2021 = {}
prueba["PRED_REFERENCIA"] = prueba["HIST_CLONE_CROP"].fillna(prueba["HIST_CLONE"]).fillna(prueba["HIST_LOC"]).fillna(entrenamiento["T_SPACRE"].mean())
evaluacion_final = []
predicciones_finales = {"Referencia histórica": prueba["PRED_REFERENCIA"].to_numpy()}
for nombre, algoritmo in algoritmos.items():
    modelo = clone(algoritmo)
    modelo.fit(entrenamiento[entradas], entrenamiento["T_SPACRE"])
    modelos_2021[nombre] = modelo
    prediccion = modelo.predict(prueba[entradas])
    predicciones_finales[nombre] = prediccion
    evaluacion_final.append({"MODELO": nombre, "MAE": mean_absolute_error(prueba["T_SPACRE"], prediccion), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prediccion)), "R2": r2_score(prueba["T_SPACRE"], prediccion)})
evaluacion_final.append({"MODELO": "Referencia histórica", "MAE": mean_absolute_error(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"]), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"])), "R2": r2_score(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"])})
prediccion_reciente = referencia_reciente(entrenamiento, prueba)
predicciones_finales["Referencia clon-ambiente reciente"] = prediccion_reciente.to_numpy()
evaluacion_final.append({"MODELO": "Referencia clon-ambiente reciente", "MAE": mean_absolute_error(prueba["T_SPACRE"], prediccion_reciente), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prediccion_reciente)), "R2": r2_score(prueba["T_SPACRE"], prediccion_reciente)})
evaluacion_final = pd.DataFrame(evaluacion_final)
evaluacion_final["ANIO_EVALUACION"] = 2021
evaluacion_final["MAX_ANIO_ENTRENAMIENTO"] = int(entrenamiento["YEAR"].max())
display(evaluacion_final.round(2))
prueba["MODELO_ELEGIDO"] = elegido
prueba["PREDICCION"] = modelos_2021[elegido].predict(prueba[entradas])
prueba["ERROR_ABSOLUTO"] = (prueba["PREDICCION"] - prueba["T_SPACRE"]).abs()
prueba["CLON_CON_HISTORIA"] = prueba["CLONE"].isin(entrenamiento["CLONE"])
print("Error del modelo elegido por disponibilidad de historia del clon:")
display(prueba.groupby("CLON_CON_HISTORIA").agg(n=("ERROR_ABSOLUTO", "size"), mae=("ERROR_ABSOLUTO", "mean")).round(2))
print("Error por ciclo y etapa:")
display(prueba.groupby(["STAGE", "CROP"]).agg(n=("ERROR_ABSOLUTO", "size"), mae=("ERROR_ABSOLUTO", "mean")).round(2))
print("MAE en etapa 5:", round(prueba.loc[prueba["STAGE"].eq("5"), "ERROR_ABSOLUTO"].mean(), 2))
errores_ambiente = prueba.assign(SESGO=prueba["PREDICCION"] - prueba["T_SPACRE"]).groupby(["LOC", "STAGE"]).agg(
    n=("ERROR_ABSOLUTO", "size"), mae=("ERROR_ABSOLUTO", "mean"), sesgo=("SESGO", "mean")
).reset_index()
print("Error por localidad y etapa; considerar el número de observaciones de cada ambiente:")
display(errores_ambiente.sort_values("mae", ascending=False).round(2))
errores_ambiente.to_csv(RUTA.parent / "serie1_errores_ambiente_2021.csv", index=False)
prueba[["FILA_CSV", "MODELO_ELEGIDO", "CLONE", "LOC", "YEAR", "STAGE", "CROP", "T_SPACRE", "PREDICCION", "PRED_REFERENCIA", "ERROR_ABSOLUTO", "CLON_CON_HISTORIA"]].to_csv(RUTA.parent / "serie1_predicciones_2021.csv", index=False)
evaluacion_final.to_csv(RUTA.parent / "serie1_evaluacion_2021.csv", index=False)
pd.concat([
    pd.DataFrame({
        "FILA_CSV": prueba["FILA_CSV"].to_numpy(), "OBSERVADO": prueba["T_SPACRE"].to_numpy(),
        "PREDICCION": prediccion, "MODELO": nombre, "ANIO_EVALUACION": 2021,
        "MAX_ANIO_ENTRENAMIENTO": int(entrenamiento["YEAR"].max()),
    }) for nombre, prediccion in predicciones_finales.items()
], ignore_index=True).to_csv(RUTA.parent / "serie1_predicciones_modelos_2021.csv", index=False)


# Sensibilidad del entrenamiento: mismas entradas, algoritmo y prueba limpia.
con_alertas = agregar_historia(datos.loc[datos["T_SPACRE"].notna()])
con_alertas[categoricas] = con_alertas[categoricas].astype(str)
modelo_sensible = clone(algoritmos[elegido])
antes_2021 = con_alertas.loc[con_alertas["YEAR"].le(2020)]
modelo_sensible.fit(antes_2021[entradas], antes_2021["T_SPACRE"])
# Se comparan en exactamente las mismas filas y con historia de la alternativa.
misma_prueba = con_alertas.loc[con_alertas["FILA_CSV"].isin(prueba["FILA_CSV"])].set_index("FILA_CSV").loc[prueba["FILA_CSV"]]
print("MAE con alertas incluidas en entrenamiento e historia, sobre las mismas filas de prueba:", round(mean_absolute_error(misma_prueba["T_SPACRE"], modelo_sensible.predict(misma_prueba[entradas])), 2))
print("MAE del escenario principal, sin ceros en la prueba:", round(prueba.loc[prueba["T_SPACRE"].gt(0), "ERROR_ABSOLUTO"].mean(), 2))

,MODELO,MAE,RMSE,R2,ANIO_EVALUACION,MAX_ANIO_ENTRENAMIENTO
0,Ridge interpretable,2144.55,2770.97,0.25,2021,2020
1,Árbol de decisión,2330.43,2983.90,0.13,2021,2020
2,Bosque aleatorio,2261.69,2877.42,0.19,2021,2020
3,Referencia histórica,2662.69,3385.62,-0.12,2021,2020
4,Referencia clon-ambiente reciente,2270.33,2938.43,0.16,2021,2020


Error del modelo elegido por disponibilidad de historia del clon:


,n,mae
CLON_CON_HISTORIA,,
False,308,2173.05
True,1980,2140.11


Error por ciclo y etapa:


n      mae
STAGE CROP              
3     0     449  2839.22
      1     188  2457.20
      2      84  1779.53
      3      28  2672.95
4     0      76  1729.03
      1      48  2675.20
5     0     382  1800.81
      1     420  1945.58
      2     335  2088.91
      3     248  1717.61
      4      30  1833.80

MAE en etapa 5: 1898.11
Error por localidad y etapa; considerar el número de observaciones de cada ambiente:


,LOC,STAGE,n,mae,sesgo
13,MAGNOLIA,5,72,3498.54,3498.54
15,MEL,3,80,3372.01,-2321.51
16,NEW,3,81,3084.16,-2363.58
21,WES,3,80,2998.28,-1176.88
1,AFH,4,38,2897.00,2685.92
0,AFH,3,136,2688.59,751.94
11,LANAUX,5,150,2669.40,2666.39
2,AFL,3,108,2586.00,719.61
4,ALMA,5,109,2259.51,2230.52
14,MARY,5,102,2251.13,1742.67


MAE con alertas incluidas en entrenamiento e historia, sobre las mismas filas de prueba: 2135.1
MAE del escenario principal, sin ceros en la prueba: 2144.55


## Interpretabilidad y uso gerencial

El árbol muestra reglas de planificación sobre las entradas disponibles antes de cosechar. Ridge permite explicar un efecto aditivo condicionado a las otras variables; sus coeficientes regularizados no deben confundirse con efectos causales. El algoritmo con menor MAE temporal se propone para planificación, condicionado a que mejore a la referencia histórica y a su desempeño en etapa 5.

Una predicción puntual sirve para comparar escenarios y presupuestar producción esperada; no asegura el resultado de una parcela. El error observado en 2021 debe acompañar la cifra y desglosarse por ambiente. Para obtener intervalos de predicción con cobertura comprobada hacen falta calibración temporal adicional y revisión específica; MAE por sí solo no es un intervalo de confianza.

En el ajuste Ridge evaluado en 2021, manteniendo las demás entradas fijas, `CROP` se asocia con aproximadamente 802 lb/acre menos por unidad y `YEAR` con 25 lb/acre menos por año. Son coeficientes regularizados y condicionados a los otros predictores; no prueban una caída causal constante por soca ni bastan para decidir cuándo renovar. Las medias históricas del clon, la localidad y su combinación con el corte aportan contexto previo.

En las validaciones temporales, el sesgo se calcula como `predicción − observado`: Ridge sobreestima en promedio 2,148 lb/acre en 2019 y subestima 1,078 lb/acre en 2020. La caída del objetivo en 2019 y su rebote en 2020 hacen plausible que un modelo entrenado con años previos quede alto en la caída y tarde en recuperar el nivel; esto es una explicación compatible con la estructura del pronóstico, no una causa probada. El contexto externo disponible relaciona 2019 con una cosecha previa muy húmeda y condiciones meteorológicas adversas, y documenta cambios operativos del programa durante COVID en 2020. Estos datos no permiten separar clima, operación, manejo, cobertura ni medición, y las combinaciones coincidentes entre años no son parcelas pareadas. La tabla temporal muestra el sesgo por algoritmo y año.


In [4]:
arbol = modelos_2021["Árbol de decisión"]
nombres = arbol.named_steps["preparar"].get_feature_names_out().tolist()
print(export_text(arbol.named_steps["regresor"], feature_names=nombres, max_depth=3))
mae_referencia = evaluacion_final.loc[evaluacion_final["MODELO"].eq("Referencia histórica"), "MAE"].iloc[0]
mae_elegido = evaluacion_final.loc[evaluacion_final["MODELO"].eq(elegido), "MAE"].iloc[0]
print(f"Algoritmo elegido: {elegido}; MAE 2021: {mae_elegido:.2f} lb/acre; mejora frente a referencia: {100 * (1 - mae_elegido / mae_referencia):.1f} %.")
print("Verificación de particiones, referencias y métricas: python -m pytest tests/integracion -q")
ridge = modelos_2021["Ridge interpretable"]
preparador = ridge.named_steps["preparar"]
coeficientes = ridge.named_steps["regresor"].coef_
nombres = preparador.get_feature_names_out()
tabla_coeficientes = pd.DataFrame({"VARIABLE": nombres, "COEFICIENTE": coeficientes})
print("Coeficientes numéricos de Ridge por unidad original de entrada:")
coef_numericos = tabla_coeficientes.loc[tabla_coeficientes["VARIABLE"].str.startswith("numeros__")].copy()
coef_numericos["COEFICIENTE_POR_UNIDAD"] = coef_numericos["COEFICIENTE"].to_numpy() / preparador.named_transformers_["numeros"].named_steps["escalar"].scale_
display(coef_numericos[["VARIABLE", "COEFICIENTE_POR_UNIDAD"]].round(3))
print("Son asociaciones condicionadas y regularizadas; no representan efectos causales de cambiar año o corte.")
mae_reciente = evaluacion_final.loc[evaluacion_final["MODELO"].eq("Referencia clon-ambiente reciente"), "MAE"].iloc[0]
print(f"Mejora frente a referencia clon-ambiente reciente en 2021: {100 * (1 - mae_elegido / mae_reciente):.1f} %.")
print(f"Error absoluto agregado / azúcar observada en 2021: {100 * prueba['ERROR_ABSOLUTO'].sum() / prueba['T_SPACRE'].sum():.1f} %.")
print(f"Equivalente del MAE en ingreso bruto a US$0.20/lb: US${mae_elegido * 0.20:.2f}/acre; no es un intervalo de confianza ni el error del aporte económico total.")


|--- numeros__HIST_LOC <= 0.11
|   |--- numeros__CROP <= 0.55
|   |   |--- categorias__LOC_AFL <= 0.50
|   |   |   |--- numeros__HIST_CLONE_CROP <= 1.24
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- numeros__HIST_CLONE_CROP >  1.24
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- categorias__LOC_AFL >  0.50
|   |   |   |--- categorias__STAGE_2 <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- categorias__STAGE_2 >  0.50
|   |   |   |   |--- truncated branch of depth 3
|   |--- numeros__CROP >  0.55
|   |   |--- numeros__HIST_LOC <= -0.60
|   |   |   |--- categorias__LOC_MAGNOLIA <= 0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- categorias__LOC_MAGNOLIA >  0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- numeros__HIST_LOC >  -0.60
|   |   |   |--- categorias__LOC_AFL <= 0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- categorias__LOC_AFL >  0.50
|   |   |   |   |--- v

,VARIABLE,COEFICIENTE_POR_UNIDAD
631,numeros__YEAR,-25.069
632,numeros__CROP,-801.670
633,numeros__HIST_CLONE,0.013
634,numeros__HIST_LOC,-0.216
635,numeros__HIST_CLONE_CROP,0.058


Son asociaciones condicionadas y regularizadas; no representan efectos causales de cambiar año o corte.
Mejora frente a referencia clon-ambiente reciente en 2021: 5.5 %.
Error absoluto agregado / azúcar observada en 2021: 25.5 %.
Equivalente del MAE en ingreso bruto a US$0.20/lb: US$428.91/acre; no es un intervalo de confianza ni el error del aporte económico total.


## Modelo para la siguiente zafra disponible

Después de conservar la evaluación de 2021, se reajusta el algoritmo elegido con todos los datos disponibles hasta 2021. La siguiente entrada ilustra el uso para 2022, una zafra posterior a la última del archivo. No es un pronóstico de una zafra actual ni una observación evaluada: su objetivo es mostrar cómo generar entradas e historia disponibles antes de cosechar.

In [5]:
modelo_planificacion = clone(algoritmos[elegido])
modelo_planificacion.fit(modelado[entradas], modelado["T_SPACRE"])
localidad_ejemplo = base.loc[base["STAGE"].eq(5), "LOC"].mode().iloc[0]
entrada = pd.DataFrame([{"CLONE": "HOCP14-885", "LOC": localidad_ejemplo, "STAGE": 5, "YEAR": 2022, "CROP": 2, "T_SPACRE": np.nan}])
con_futuro = pd.concat([base, entrada], ignore_index=True)
ejemplo = agregar_historia(con_futuro).tail(1).copy()
ejemplo[categoricas] = ejemplo[categoricas].astype(str)
ejemplo["PREDICCION_LB_ACRE"] = modelo_planificacion.predict(ejemplo[entradas])
display(ejemplo[["CLONE", "LOC", "YEAR", "STAGE", "CROP", "PREDICCION_LB_ACRE"]].round(2))
print("Predicción ilustrativa; su resultado real no está en el archivo.")

,CLONE,LOC,YEAR,STAGE,CROP,PREDICCION_LB_ACRE
28981,HOCP14-885,RHEBERT,2022,5,2,9129.8


Predicción ilustrativa; su resultado real no está en el archivo.
